In [1]:
import librosa
import numpy as np
import os
# Set the duration and hop length for audio processing
duration = 5 # seconds
hop_length = 512

# Define a function to load and extract LFCC features from an audio file
def extract_mfcc_features(file_path):
    # Load the audio file and extract the LFCC features
    y, sr = librosa.load(file_path, duration=duration)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13, hop_length=hop_length)
    return mfcc

# Define a function to load and preprocess the dataset
def load_dataset(dataset_path):
    # Load the audio files and extract the LFCC features
    mfcc_features = []
    labels = []
    for label, class_name in enumerate(sorted(os.listdir(dataset_path))):
        class_path = os.path.join(dataset_path, class_name)
        for file_name in os.listdir(class_path):
            file_path = os.path.join(class_path, file_name)
            mfcc = extract_mfcc_features(file_path)
            mfcc_features.append(mfcc)
            labels.append(label)
    # Convert the LFCC features and labels to numpy arrays
    mfcc_features = np.array(mfcc_features)
    labels = np.array(labels)
    # Normalize the LFCC features
    mean = np.mean(mfcc_features, axis=0)
    std = np.std(mfcc_features, axis=0)
    np.save('mean.npy', mean)
    np.save('std.npy', std)
    mfcc_features -= np.mean(mfcc_features, axis=0)
    mfcc_features /= np.std(mfcc_features, axis=0)
    
    return mfcc_features, labels


# Load the dataset
dataset_path = '/kaggle/input/detect-cry/data'
mfcc_featurs, labes = load_dataset(dataset_path)

In [4]:
import numpy as np
from sklearn.model_selection import train_test_split
from keras.models import Sequential
from keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPooling2D
from keras.utils import to_categorical

# Load the data
#lfcc_features = np.load('lfcc_features.npy')
#labels = np.load('labels.npy')

# Normalize the data
mfcc_featurs /= np.std(mfcc_featurs, axis=0)
num_classes=3
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(mfcc_featurs, labes, test_size=0.2, random_state=42)

# Convert the labels to categorical format
y_train = to_categorical(y_train)
y_test = to_categorical(y_test)
#X_train = X_train.reshape(X_train.shape[0], 13, 87, 1)
#X_test = X_test.reshape(X_test.shape[0], 13, 87, 1)
# Define the CNN architecture
model = Sequential()
model.add(Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=(13, 216, 1)))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Conv2D(64, kernel_size=(3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Flatten())
model.add(Dense(128, activation='relu'))
model.add(Dense(num_classes, activation='softmax'))

# Compile the model
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Train the model
epochs = 100
batch_size = 32
history = model.fit(X_train, y_train, batch_size=batch_size, epochs=epochs, validation_data=(X_test, y_test))

# Evaluate the model on the test set
score = model.evaluate(X_test, y_test, verbose=0)
print('Test loss:', score[0])
print('Test accuracy:', score[1])
model.summary()
model.save('classification.h5');

Epoch 1/100
5/5 [==============================] - 2s 120ms/step - loss: 0.7535 - accuracy: 0.5556 - val_loss: 0.5681 - val_accuracy: 0.6667
Epoch 2/100
5/5 [==============================] - 0s 55ms/step - loss: 0.4105 - accuracy: 0.7847 - val_loss: 0.4212 - val_accuracy: 0.6667
Epoch 3/100
5/5 [==============================] - 0s 57ms/step - loss: 0.3417 - accuracy: 0.8194 - val_loss: 0.3576 - val_accuracy: 0.8333
Epoch 4/100
5/5 [==============================] - 0s 52ms/step - loss: 0.2805 - accuracy: 0.8750 - val_loss: 0.2602 - val_accuracy: 0.9444
Epoch 5/100
5/5 [==============================] - 0s 55ms/step - loss: 0.1902 - accuracy: 0.9444 - val_loss: 0.2332 - val_accuracy: 0.8889
Epoch 6/100
5/5 [==============================] - 0s 57ms/step - loss: 0.1167 - accuracy: 0.9931 - val_loss: 0.1469 - val_accuracy: 0.9722
Epoch 7/100
5/5 [==============================] - 0s 52ms/step - loss: 0.0682 - accuracy: 1.0000 - val_loss: 0.0691 - val_accuracy: 1.0000
Epoch 8/100
5/5 [==

In [12]:
    import librosa
    import numpy as np
    import requests
    from keras.models import load_model
    model = load_model('/kaggle/working/classification.h5')
    y, sr = librosa.load('/kaggle/input/detect-cry/data/Silence/silence.wav_107.wav', sr=16000)
    # Extract MFCC features
    mfcc_features = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13, n_fft=1024, hop_length=372)
    mfcc_features = mfcc_features.T

    # Reshape data for use with the model
    #num_frames = 13  # change this to match the number of frames in your data
    #num_features = 87
    print(mfcc_features.shape)
    mfcc_features = mfcc_features.reshape(1, 13, 216, 1)

    # Make a prediction using the trained model
    prediction = model.predict(mfcc_features)
    #prediction_class = np.argmax(prediction)

    # Print the predicted class
    #classes = ['Hunger', 'Discomfort', 'Sleepiness', 'Pain', 'None']
    #print('Predicted class:', classes[prediction_class])
    print(prediction)
    class_labels = ['cry','Silence','laugh']
    predicted_label = class_labels[np.argmax(prediction)]

    # Print the predicted class label
    print('The predicted reason for the baby cry is:', predicted_label)
    


(216, 13)
1/1 [==============================] - 0s 99ms/step
[[0. 0. 1.]]
The predicted reason for the baby cry is: laugh
